In [3]:
%pip install transformers bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.


In [22]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA H200 NVL


In [5]:
import torch
torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [17]:
prompt0 =  """
You are a careful researcher who generates clean, realistic synthetic statistical tables.
Generate exactly one synthetic statistical table in valid CSV format.

Requirements:
1. The table must have between 5 and 10 columns.
2. The table must contain exactly 15 data rows, plus 1 header row.
3. Invent realistic column names. Include a mix of:
   - numeric columns (integers and/or floats)
   - Numeric columns must contain only plain numbers (no $, commas, or % symbols)
   - categorical columns
4. Make the data look plausible for a real-world dataset.
5. Ensure values are varied across rows.
6. All rows must have the same number of columns as the header.
7. Do not include commas inside cell values.
8. Do not include explanations, markdown, code fences, titles, or any text before or after the CSV.
9. Output only the CSV table.

The dataset can be about any realistic domain such as health, education, business, transportation, or demographics.

Here is an example table in CSV format:
id,gender,age,height_cm,weight_kg,bmi,annual_income_$k,sleep_hours,resting_hr_bpm,avg_steps_day
P001,man,26,177.5,68.4,21.7,62.9,7.3,70,5542
P002,woman,42,151.2,55.1,24.1,50.9,7.3,58,6695
P003,woman,27,155.0,64.9,27.0,63.1,7.0,81,9534
P004,woman,21,164.2,80.6,29.9,20.1,6.8,78,4423
P005,man,56,181.2,76.5,23.3,53.2,5.6,78,8069
P006,man,35,177.2,77.6,24.7,73.0,6.6,61,10768
P007,man,39,175.2,62.9,20.5,135.3,6.7,84,3481
P008,woman,21,161.2,52.2,20.1,45.1,6.2,56,7962
P009,woman,31,154.1,69.3,29.2,38.5,6.8,76,8150
P010,woman,39,158.7,79.6,31.6,45.5,7.4,92,9455
P011,man,24,172.8,75.2,25.2,99.3,8.9,60,4408
P012,woman,43,169.3,86.0,30.0,71.9,7.2,64,4199
P013,woman,31,165.1,73.9,27.1,44.8,7.3,71,8805
P014,man,34,163.7,60.6,22.6,79.6,6.9,65,8242
P015,man,31,178.3,86.2,27.1,63.3,5.1,54,8126
P016,man,60,173.3,97.3,32.4,102.2,7.0,71,8366
P017,man,38,171.3,74.2,25.3,40.8,7.1,59,5800
P018,woman,25,166.7,90.3,32.5,50.1,9.5,75,8081
P019,man,48,183.2,58.7,17.5,48.4,6.8,61,8233
P020,man,23,182.5,97.3,29.2,26.8,7.3,85,5714
"""

In [18]:
from transformers import pipeline
import re
import os
from io import StringIO
import csv

In [19]:
def generate_table(model, model_name, tokenizer, counter):
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer
    )
    
    generation = generator(
        prompt0,
        do_sample=False,
        max_new_tokens=5000,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    print(generation[0]["generated_text"])

    full_text = generation[0]["generated_text"]
    full_text = re.sub(r"<think>.*?</think>", "", full_text, flags=re.DOTALL).strip()
    print(full_text)

    folder_path = f"table_tests/{model_name}/"
    os.makedirs(folder_path, exist_ok=True)
    
    # Assume full_text contains your LLM CSV output
    candidate_tables = StringIO(full_text)
    reader = csv.reader(candidate_tables)
    
    # Read all rows
    all_rows = [r for r in reader if any(cell.strip() for cell in r)]  # remove empty rows
    
    if not all_rows:
        print("No rows found")
    else:
        header = all_rows[0]       # first non-empty row is header
        rows = all_rows[1:]        # the rest are data rows
    
        # Ensure all rows have the same number of columns as header
        num_cols = len(header)
        valid_rows = [r for r in rows if len(r) == num_cols]
    
        # Combine header + valid rows for saving
        table_cleaned = [header] + valid_rows
    
        # Save CSV
        file_name = f"table_{counter}.csv"
        full_path = os.path.join(folder_path, file_name)
        with open(full_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerows(table_cleaned)
    
        print(f"Saved {file_name} with {len(valid_rows)} rows")
        print("Header:", header)
        print("First few rows:", valid_rows[:5])

In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

load gemma
gemma_name = "google/gemma-4-31B"

gemma = AutoModelForCausalLM.from_pretrained(
    gemma_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
)

gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_name)

for i in range(0, 5):
    generate_table(gemma, gemma_name, gemma_tokenizer, i)

del gemma  # or your large tensor
gc.collect()
torch.cuda.empty_cache()

# load llama
llama_name = "meta-llama/Llama-3.1-70B-Instruct"

llama = AutoModelForCausalLM.from_pretrained(
    llama_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
)

llama_tokenizer = AutoTokenizer.from_pretrained(llama_name)

for i in range(0, 5):
    generate_table(llama, llama_name, llama_tokenizer, i)

del llama  # or your large tensor
gc.collect()
torch.cuda.empty_cache()

# load qwen
qwen_name = "Qwen/Qwen3-32B"

qwen = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)

for i in range(0, 5):
    generate_table(qwen, qwen_name, qwen_tokenizer, i)
    
del qwen  # or your large tensor
gc.collect()
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

KeyboardInterrupt: 